In [ ]:
# 读取csv
import pandas as pd

online_txt_data = pd.read_csv('国学梦.csv')
online_txt_data

In [ ]:

import json
import requests
from bs4 import BeautifulSoup

book_json = []

for index,charpter,url in online_txt_data.itertuples(index=False):
    print(f'正在抓取第 {index} 章: {charpter} ......')
    response = requests.get(url)
    response.encoding = 'utf-8'

    soup = BeautifulSoup(response.text, 'html.parser')

    # 提取div lacontent底下的所有 p 标签
    lacontent_div = soup.find('div', class_='lacontent')
    p_tags = lacontent_div.find_all('p')
    
    # 去除最后一个 p 标签
    p_tags = p_tags[:-1]

    # 提取文本内容并合并为一个字符串
    content = '\n'.join(p.get_text() for p in p_tags)
    book_json.append({
        'index': index,
        'charpter': charpter,
        'content': content
    })
    
with open('国学梦-西游记白话版.json', 'w', encoding='utf-8') as f:
    json.dump(book_json, f, ensure_ascii=False, indent=4)
        

In [ ]:
# 结巴分词
import jieba
import re

json_files = ['国学梦-西游记白话版.json', '汉程网-西游记白话版.json']


# book_content = json.load(open("西游记-原文.json", 'r', encoding='utf-8'))
# book_content = json.load(open('汉程网-西游记白话版.json', 'r', encoding='utf-8'))
book_content = json.load(open('国学梦-西游记白话版.json', 'r', encoding='utf-8'))

# print(book_content[0])

charpter_content=book_content[0]['content']

# 正则表达式去除标点符号、空字符
charpter_content = re.sub(r'[^\w\s]', '', charpter_content)

# 结巴分词不同模式
mode = [("精确模式",jieba.cut(charpter_content, cut_all=False)),
        ("全模式",jieba.cut(charpter_content, cut_all=True)),
        ("搜索引擎模式",jieba.cut_for_search(charpter_content))]

# stopwords_cn.txt
stop_words = set()
with open('stopwords_cn.txt', 'r', encoding='utf-8') as f:
    for line in f:
        stop_words.add(line.strip())
        
        
for m,result in mode:
    # 统计词频
    word_freq = {}
    for word in result:
        if word not in stop_words and word.strip() != '':
            word_freq[word] = word_freq.get(word, 0) + 1
    # 按词频排序
    sorted_word_freq = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)
    print(f"\n{m} , {len(sorted_word_freq)} 个不同的词语, 词频前10名:")
    for word, freq in sorted_word_freq[:10]:
        print(f"{word}: {freq}")
    



精确模式 , 474 个不同的词语, 词频前10名:
祖师: 20
孙悟空: 19
猴子: 12
美猴王: 9
走: 8
悟空: 8
菩提: 7
石猴: 6
师父: 6
长生不老: 5

全模式 , 585 个不同的词语, 词频前10名:
悟空: 27
祖师: 20
孙悟空: 19
石: 14
猴: 13
猴子: 12
猴王: 12
美猴王: 9
菩提: 7
师父: 6

搜索引擎模式 , 550 个不同的词语, 词频前10名:
悟空: 27
祖师: 20
孙悟空: 19
猴子: 12
猴王: 12
美猴王: 9
走: 8
菩提: 7
石猴: 6
师父: 6


In [ ]:
with open('hcw.txt',    'r', encoding='utf-8') as f:
    text = f.read()
    with open('汉程网-西游记白话版.json', 'w', encoding='utf-8') as out_f:
        json.dump([{
            "index": 1,
        "charpter": "惊天地美猴王出世",
        'content': text}], out_f, ensure_ascii=False, indent=4)

In [ ]:
# 分割西游记原文
# 通过 ”第一回　灵根育孕源流出　心性修持大道生“ 等章节标题进行分割
import json
import re

text = ''
with open('西游记.txt', 'r', encoding='utf-8') as f:
    text = f.read()
    
print(f'小说总字数：{len(text)} 字。')
    
    
# 第xx回(....)\n
chapters = re.split(r'(第[一二三四五六七八九十百零两]+回[^\n]+\n)', text)
print(f'共分割出 {len(chapters)//2} 章章节内容。')
book_json = []
for i in range(1, len(chapters), 2):
    charpter_title = chapters[i].strip()
    charpter_content = chapters[i+1].strip()
    book_json.append({
        'index': (i // 2) + 1,
        'charpter': charpter_title,
        'content': charpter_content
    })
    
with open('西游记-原文.json', 'w', encoding='utf-8') as f:
    json.dump(book_json, f, ensure_ascii=False, indent=4)

    


小说总字数：730234 字。
共分割出 100 章章节内容。


In [ ]:
from hanlp.common import HanLP

# 加载古汉语分词模型（自动下载适配语料）
tokenizer = HanLP.load(HanLP.pretrained.tok.UD_CTB7_HANLP)
# 针对明清白话的优化配置
HanLP.Config.tok = 'cn古典'

# 西游记文本示例
text = "那猴在山中，却会行走跳跃，食草木，饮涧泉，采山花，觅树果；与狼虫为伴，虎豹为群，獐鹿为友，猕猿为亲；夜宿石崖之下，朝游峰洞之中。"
# 分词结果
tokens = HanLP.tokenize(text, mode='古典')
print(tokens)
# 输出：['那猴', '在', '山中', '，', '却会', '行走跳跃', '，', '食', '草木', '，', '饮', '涧泉', '，', '采', '山花', '，', '觅', '树果', '；', ...]

ImportError: cannot import name 'HanLP' from 'hanlp' (c:\Users\lfdeng\miniconda3\envs\crawler\Lib\site-packages\hanlp\__init__.py)